# Package

In [5]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [6]:
# ============================================================
# FIX PROJECT ROOT (notebook in /3_notebook)
# ============================================================
from pathlib import Path
import pandas as pd
from utilsforecast.plotting import plot_series

# Le notebook est dans Explainable_AI_.../3_notebook
PROJECT_ROOT = Path.cwd().parent

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
print("PROJECT_ROOT =", PROJECT_ROOT)

# ============================================================
# 0) Charger les 2 modèles (parquet) depuis PROJECT_ROOT
# ============================================================
path_ar12 = PROJECT_ROOT / "outputs" / "forecasts" / "unrate_ar_lag12_oos_forecasts.parquet"
path_arp  = PROJECT_ROOT / "outputs" / "forecasts" / "unrate_ar_pstar_oos_forecasts.parquet"
path_lr = PROJECT_ROOT / "outputs" / "forecasts" / "unrate_lr_lag12_exog_oos_forecasts.parquet"

df_ar12 = pd.read_parquet(path_ar12)
df_arp  = pd.read_parquet(path_arp)
df_lr   = pd.read_parquet(path_lr)

df_ar12["date"] = pd.to_datetime(df_ar12["date"]).dt.tz_localize(None)
df_arp["date"]  = pd.to_datetime(df_arp["date"]).dt.tz_localize(None)
df_lr["date"]  = pd.to_datetime(df_lr["date"]).dt.tz_localize(None)

PROJECT_ROOT = d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA


# Plot

In [7]:
# ============================================================
# 1) Harmoniser clés (series_id/date) si besoin
# ============================================================
def ensure_keys(df):
    df = df.copy()
    if "unique_id" in df.columns and "series_id" not in df.columns:
        df = df.rename(columns={"unique_id": "series_id"})
    if "ds" in df.columns and "date" not in df.columns:
        df = df.rename(columns={"ds": "date"})
    return df

df_ar12 = ensure_keys(df_ar12)
df_arp  = ensure_keys(df_arp)
df_lr   = ensure_keys(df_lr)

# ============================================================
# 2) Sélection colonnes AR (cas standard: y_obs, y_hat_ar, y_hat_ar_lo_95, y_hat_ar_hi_95)
# ============================================================
need_ar = ["series_id", "date", "y_obs", "y_hat_ar", "y_hat_ar_lo_95", "y_hat_ar_hi_95"]
missing_ar12 = [c for c in need_ar if c not in df_ar12.columns]
missing_arp  = [c for c in need_ar if c not in df_arp.columns]

assert not missing_ar12, f"AR(12) parquet missing columns: {missing_ar12}"
assert not missing_arp,  f"AR(p*) parquet missing columns: {missing_arp}"

base_ar12 = (
    df_ar12[need_ar]
    .rename(columns={
        "y_hat_ar": "y_hat_ar12",
        "y_hat_ar_lo_95": "y_hat_ar12_lo_95",
        "y_hat_ar_hi_95": "y_hat_ar12_hi_95",
    })
)

base_arp = (
    df_arp[need_ar]
    .rename(columns={
        "y_hat_ar": "y_hat_arp",
        "y_hat_ar_lo_95": "y_hat_arp_lo_95",
        "y_hat_ar_hi_95": "y_hat_arp_hi_95",
    })
    # on évite d'avoir 2 fois y_obs après merge
    .drop(columns=["y_obs"])
)

# ============================================================
# 2bis) Sélection colonnes LR (tolérant: avec ou sans PI)
# ============================================================
need_lr_base = ["series_id", "date"]
assert all(c in df_lr.columns for c in need_lr_base), f"LR parquet missing keys: {[c for c in need_lr_base if c not in df_lr.columns]}"

# Trouver la colonne de prédiction LR (on teste plusieurs noms possibles)
lr_pred_candidates = ["y_hat_lr", "y_hat", "y_pred", "yhat", "y_hat_linreg", "y_hat_lr_exog"]
lr_pred = next((c for c in lr_pred_candidates if c in df_lr.columns), None)
assert lr_pred is not None, (
    "LR parquet: colonne de prédiction introuvable. "
    f"Testé: {lr_pred_candidates}. Colonnes dispo: {list(df_lr.columns)}"
)

# Intervalles optionnels (si présents)
lr_lo = next((c for c in ["y_hat_lr_lo_95", "y_hat_lo_95", "y_pred_lo_95", "yhat_lo_95"] if c in df_lr.columns), None)
lr_hi = next((c for c in ["y_hat_lr_hi_95", "y_hat_hi_95", "y_pred_hi_95", "yhat_hi_95"] if c in df_lr.columns), None)

cols_lr = need_lr_base + [lr_pred] + ([lr_lo] if lr_lo else []) + ([lr_hi] if lr_hi else [])
base_lr = df_lr[cols_lr].rename(columns={lr_pred: "y_hat_lr"})

# si LR contient y_obs (parfois), on le drop pour éviter doublon
if "y_obs" in base_lr.columns:
    base_lr = base_lr.drop(columns=["y_obs"])

# ============================================================
# 3) Merge : une table unique avec AR12 + ARp + LR
# ============================================================
df_ar_forecasts = (
    base_ar12
    .merge(base_arp, on=["series_id", "date"], how="inner")
    .merge(base_lr,  on=["series_id", "date"], how="inner")
)

# ============================================================
# 4) Même STRUCTURE que ton code : df_obs + df_fcst
# ============================================================
df_obs = (
    df_ar_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_obs": "y",
    })
    [["unique_id", "ds", "y"]]
)

# Forecasts : AR12 + ARp (avec PI) + LR (PI si dispo)
fcst_cols = [
    "unique_id", "ds",
    "AR12", "AR12-lo-95", "AR12-hi-95",
    "ARp",  "ARp-lo-95",  "ARp-hi-95",
    "LR",
]
rename_map = {
    "series_id": "unique_id",
    "date": "ds",

    "y_hat_ar12": "AR12",
    "y_hat_ar12_lo_95": "AR12-lo-95",
    "y_hat_ar12_hi_95": "AR12-hi-95",

    "y_hat_arp": "ARp",
    "y_hat_arp_lo_95": "ARp-lo-95",
    "y_hat_arp_hi_95": "ARp-hi-95",

    "y_hat_lr": "LR",
}

# si LR a des PI, on les ajoute automatiquement
if lr_lo and lr_hi:
    rename_map[lr_lo] = "LR-lo-95"
    rename_map[lr_hi] = "LR-hi-95"
    fcst_cols = fcst_cols[:-1] + ["LR", "LR-lo-95", "LR-hi-95"]  # insère PI juste après LR

df_fcst = (
    df_ar_forecasts
    .rename(columns=rename_map)
    [[c for c in fcst_cols if c in (set(rename_map.values()) | {"unique_id", "ds"})]]
)

# ============================================================
# 5) Plot + rename légende
# ============================================================
fig = plot_series(
    df=df_obs,
    forecasts_df=df_fcst,
    level=[95],
    engine="plotly",
).update_layout(height=400)

for trace in fig.data:
    n = (trace.name or "")
    nl = n.lower()

    if n == "y":
        trace.name = "Unemployment rate (%)"
    elif n == "AR12":
        trace.name = "AutoRegressive AR(12)"
    elif n == "ARp":
        trace.name = "AutoRegressive AR(p*)"
    elif n == "LR":
        trace.name = "Linear Regression (exog)"
    elif "level_95" in nl:
        if "ar12" in nl:
            trace.name = "AR(12) 95% Prediction Interval"
        elif "arp" in nl:
            trace.name = "AR(p*) 95% Prediction Interval"
        elif "lr" in nl:
            trace.name = "LR 95% Prediction Interval"
        else:
            trace.name = "95% Prediction Interval"

fig.show()

On voit clairement que la régression linéaire commence à mieux anticiper les crises grâce à l'ajout des variables explicatives. 

# Analyse des erreurs

In [8]:
import numpy as np
import pandas as pd
from math import sqrt, erf, isfinite
from typing import Iterable, Optional, Dict, List, Tuple

# ============================================================
# 1) DM test (sans SciPy) — HAC Bartlett
# ============================================================
def _phi(z: float) -> float:
    return 0.5 * (1.0 + erf(z / sqrt(2.0)))

def dm_pvalue(loss_diff: np.ndarray, lags: int = 0) -> float:
    x = np.asarray(loss_diff, dtype=float)
    x = x[np.isfinite(x)]
    T = x.size
    if T < 3:
        return np.nan

    dbar = x.mean()
    gamma0 = np.dot(x - dbar, x - dbar) / T
    var = gamma0

    if lags > 0:
        for k in range(1, min(lags, T - 1) + 1):
            w = 1.0 - k / (lags + 1.0)
            cov = np.dot(x[k:] - dbar, x[:-k] - dbar) / T
            var += 2.0 * w * cov

    if var <= 0:
        return np.nan

    stat = dbar / sqrt(var / T)
    p = 2.0 * (1.0 - _phi(abs(stat)))
    return max(0.0, min(1.0, p))

In [9]:
# ============================================================
# 2) Pivot "MAE (p)" comme ton exemple
# ============================================================
def make_mae_dm_pivot(
    wide: pd.DataFrame,
    segments: List[Tuple[str, Optional[str], str]],
    *,
    methods: Optional[Iterable[str]] = None,  # None -> toutes sauf true
    include_overall: bool = True,
    overall_label: str = "Ensemble",
    min_obs: int = 20,
    round_digits: int = 4,
    add_dm: bool = True,
    dm_lags: int = 11,                        # ex h-1 si h=12
) -> pd.DataFrame:

    df = wide.copy()

    # index datetime
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
        df = df.set_index("date")
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("wide doit avoir un DatetimeIndex ou une colonne 'date'.")
    df = df.sort_index()

    if "true" not in df.columns:
        raise ValueError("wide doit contenir la colonne 'true'.")

    # méthodes
    if methods is None:
        meths = [c for c in df.columns if c != "true"]
    else:
        meths = [m for m in methods if m in df.columns and m != "true"]
    if len(meths) == 0:
        return pd.DataFrame()

    # fenêtres
    full_start, full_end = df.index.min(), df.index.max()
    windows: List[Tuple[pd.Timestamp, pd.Timestamp, str]] = []
    if include_overall:
        windows.append((full_start, full_end, overall_label))
    for start, end, label in segments:
        s = pd.to_datetime(start, utc=True)
        e = pd.to_datetime(end, utc=True) if end is not None else full_end
        windows.append((s, e, label))

    rows = []  # (model, period, cell)

    for start, end, label in windows:
        sub = df.loc[start:end, ["true"] + meths].copy().dropna(subset=["true"])

        # MAE par modèle + erreurs absolues
        maes: Dict[str, float] = {}
        err_abs: Dict[str, pd.Series] = {}

        for m in meths:
            diffs = (sub["true"] - sub[m]).abs().dropna()
            err_abs[m] = diffs
            maes[m] = float(diffs.mean()) if diffs.shape[0] >= min_obs else np.nan

        finite_models = [m for m in meths if isfinite(maes.get(m, np.nan))]
        best_m = min(finite_models, key=lambda k: maes[k]) if finite_models else None

        for m in meths:
            mae_val = maes.get(m, np.nan)
            if not isfinite(mae_val):
                rows.append((m, label, np.nan))
                continue

            cell = f"{mae_val:.{round_digits}f}"

            if add_dm and best_m is not None and m != best_m:
                v1 = err_abs[m]
                v2 = err_abs[best_m]
                common = v1.index.intersection(v2.index)
                diff = (v1.loc[common] - v2.loc[common]).to_numpy()
                if diff.size >= min_obs:
                    pval = dm_pvalue(diff, lags=dm_lags)
                    if isfinite(pval):
                        cell = f"{cell} ({pval:.3f})"

            rows.append((m, label, cell))

    out = pd.DataFrame(rows, columns=["model", "period", "value"])
    pivot = out.pivot(index="model", columns="period", values="value")

    desired_cols = ([overall_label] if include_overall else []) + [lbl for _, _, lbl in segments]
    pivot = pivot.reindex(columns=desired_cols)

    pivot = pivot.reindex(index=meths)

    pivot.columns.name = "period"
    pivot.index.name = "model"
    return pivot

In [12]:
# ============================================================
# 3) Construire wide depuis df_ar_forecasts (AR12 + ARP + LR)
# ============================================================
needed_cols = ["date", "y_obs", "y_hat_ar12", "y_hat_arp", "y_hat_lr"]
missing = [c for c in needed_cols if c not in df_ar_forecasts.columns]
assert not missing, f"df_ar_forecasts missing columns: {missing}"

wide = (
    df_ar_forecasts
    .assign(date=pd.to_datetime(df_ar_forecasts["date"], utc=True, errors="coerce"))
    .rename(columns={
        "y_obs": "true",
        "y_hat_ar12": "AR12",
        "y_hat_arp": "ARP",
        "y_hat_lr": "LR",
    })
    [["date", "true", "AR12", "ARP", "LR"]]
)

# ============================================================
# 4) Tes segments EXACTS
# ============================================================
segments = [
    ("1990-01-01", "1999-12-31", "1990-1999"),
    ("2000-01-01", "2008-12-31", "2000-2008"),
    ("2008-01-01", "2019-12-31", "2008-2019"),
    ("2020-01-01", None,         "2020-2025"),
]

methods_used = ["AR12", "ARP", "LR"]

table_pivot = make_mae_dm_pivot(
    wide=wide,
    segments=segments,
    methods=methods_used,
    include_overall=True,
    overall_label="Ensemble",
    min_obs=20,
    round_digits=4,
    add_dm=True,
    dm_lags=11,   # h-1 si h=12
)

print("Méthodes utilisées :", methods_used)
print(table_pivot)

Méthodes utilisées : ['AR12', 'ARP', 'LR']
period        Ensemble       1990-1999       2000-2008       2008-2019  \
model                                                                    
AR12    0.7043 (0.237)  0.4259 (0.091)  0.3836 (0.859)          0.6050   
ARP             0.6439          0.3118          0.3742  0.6382 (0.772)   
LR      0.8098 (0.002)  0.5002 (0.016)  0.4603 (0.328)  0.7181 (0.156)   

period       2020-2025  
model                   
AR12    1.8605 (0.056)  
ARP             1.6461  
LR      2.0331 (0.005)  


Les résultats de la régression linéaire est nettement très faible par rapport aux modèles Auto-régressif.